# Linguagens de Programação para Engenharia de Dados
## Encontro 5 — Notebook Prático: Qualidade e confiabilidade de dados

**Instituição:** Universidade de Fortaleza (UNIFOR)
**Curso:** Pós-Graduação — Especialização em Engenharia de Dados
**Professor:** Cassio Pinheiro — [LinkedIn](https://www.linkedin.com/in/cassio-pinheiro-9baa7939/)
**Data:** 10/07/2026

---

Hoje o nosso pipeline de vendas ganha uma camada de **validação e quarentena**. Partimos do *vendas tratado* (recriado aqui com pandas, de forma determinística), **medimos** sua qualidade, **validamos** com **Pandera**, forçamos uma **falha proposital** capturada, **separamos** as linhas ruins em quarentena com o motivo do erro, e consolidamos um **cartão de saúde** de observabilidade. **Great Expectations** aparece em `try/except` — o notebook roda inteiro sem depender dele.

> **Como usar:** rode as células em ordem. Todo o núcleo (pandas + Pandera) é determinístico e roda sem erro — a falha do Pandera é *capturada* de propósito, não propagada.

## Setup

Núcleo **obrigatório**: `pandas` e `pandera`. A partir da versão 0.20 o import recomendado para pandas é `import pandera.pandas as pa`; fazemos um *fallback* para `import pandera as pa` em versões mais antigas. **Great Expectations é opcional** e tratado em `try/except` mais adiante.

In [1]:
import pandas as pd

# Pandera: import tolerante a versão (0.20+ usa pandera.pandas)
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa

print("pandas :", pd.__version__)
print("pandera:", pa.__version__ if hasattr(pa, "__version__") else "ok")

pandas : 2.3.3
pandera: ok


---

# 0 — O vendas tratado (ponto de partida)

> 📖 **No `.md`:** retomamos o pipeline de vendas dos encontros anteriores.

Recriamos aqui o **vendas tratado** que vínhamos construindo desde o E1/E2 — mas de propósito **com alguns defeitos plantados**, porque o objetivo de hoje é justamente *pegar* esses defeitos. O DataFrame é fixo (sem aleatoriedade): toda execução vê exatamente as mesmas linhas, defeitos inclusive.

In [2]:
# Vendas "tratado", porém com defeitos plantados de propósito para a aula.
vendas = pd.DataFrame([
    {"id": 1001, "data": "2026-07-01", "categoria": "eletronicos",  "valor": 250.00, "cliente_id": "C001"},
    {"id": 1002, "data": "2026-07-01", "categoria": "alimentacao",  "valor":  45.50, "cliente_id": "C002"},
    {"id": 1003, "data": "2026-07-02", "categoria": "transporte",   "valor":  19.90, "cliente_id": "C003"},
    {"id": 1004, "data": "2026-07-02", "categoria": "eletronicos",  "valor": 999.00, "cliente_id": None},   # cliente ausente
    {"id": 1005, "data": "2026-07-03", "categoria": "alimentacao",  "valor":  12.00, "cliente_id": "C005"},
    {"id": 1006, "data": "2026-07-03", "categoria": "transporte",   "valor": -30.00, "cliente_id": "C006"},  # valor negativo (invalido)
    {"id": 1007, "data": "03/07/2026", "categoria": "eletronicos",  "valor": 120.00, "cliente_id": "C007"},  # data fora do padrao
    {"id": 1008, "data": "2026-07-04", "categoria": "viagem",       "valor":  80.00, "cliente_id": "C008"},  # categoria fora do dominio
    {"id": 1009, "data": "2026-07-04", "categoria": "alimentacao",  "valor":  60.00, "cliente_id": "C009"},
    {"id": 1003, "data": "2026-07-05", "categoria": "transporte",   "valor":  22.00, "cliente_id": "C010"},  # id duplicado (1003)
])

print(f"vendas tratado: {len(vendas)} linhas, {vendas.shape[1]} colunas")
vendas

vendas tratado: 10 linhas, 5 colunas


,id,data,categoria,valor,cliente_id
0,1001,2026-07-01,eletronicos,250.0,C001
1,1002,2026-07-01,alimentacao,45.5,C002
2,1003,2026-07-02,transporte,19.9,C003
3,1004,2026-07-02,eletronicos,999.0,None
4,1005,2026-07-03,alimentacao,12.0,C005
5,1006,2026-07-03,transporte,-30.0,C006
6,1007,03/07/2026,eletronicos,120.0,C007
7,1008,2026-07-04,viagem,80.0,C008
8,1009,2026-07-04,alimentacao,60.0,C009
9,1003,2026-07-05,transporte,22.0,C010


---

# 1 — Medindo qualidade

> 📖 **No `.md`:** seção *1 — Dimensões de qualidade*.

Antes de validar, **medimos**. "Dado bom" precisa virar número. Calculamos três dimensões direto sobre a tabela: **completude** (por coluna), **unicidade** (duplicatas na chave `id`) e **validade** (taxa de linhas que passam numa regra de negócio simples, `valor >= 0`).

In [3]:
# --- Completude: % de celulas nao-nulas por coluna ---
completude = (vendas.notna().mean() * 100).round(1)
print("COMPLETUDE por coluna (%):")
for col, pct in completude.items():
    print(f"  {col:<12}: {pct:5.1f}%")

# --- Unicidade: duplicatas na chave primaria 'id' ---
n_dup_id = int(vendas["id"].duplicated().sum())
print(f"\nUNICIDADE: {n_dup_id} id(s) duplicado(s) -> "
      f"{'OK' if n_dup_id == 0 else 'VIOLADO'}")

# --- Validade: taxa de linhas com valor >= 0 ---
taxa_valor_ok = (vendas["valor"] >= 0).mean() * 100
print(f"VALIDADE (valor >= 0): {taxa_valor_ok:.1f}% das linhas passam")

COMPLETUDE por coluna (%):
  id          : 100.0%
  data        : 100.0%
  categoria   : 100.0%
  valor       : 100.0%
  cliente_id  :  90.0%

UNICIDADE: 1 id(s) duplicado(s) -> VIOLADO
VALIDADE (valor >= 0): 90.0% das linhas passam


Repare: a tabela parece "tratada", mas as métricas já acusam problemas — `cliente_id` incompleto, um `id` duplicado e linhas com `valor` negativo. Medir é o primeiro passo; **validar formalmente** é o próximo.

---

# 2 — Validação com Pandera

> 📖 **No `.md`:** seção *2 — Validação programática com Pandera*.

Declaramos o que uma linha **válida** deve ser num único `DataFrameSchema`: `id` inteiro positivo e único; `valor` numérico `>= 0`; `data` no formato `AAAA-MM-DD` (regex); `categoria` dentro de um conjunto fechado; `cliente_id` obrigatório (não-nulo).

In [4]:
schema = pa.DataFrameSchema(
    {
        "id":         pa.Column(int,   pa.Check.greater_than(0), unique=True),
        "data":       pa.Column(str,   pa.Check.str_matches(r"^\d{4}-\d{2}-\d{2}$")),
        "categoria":  pa.Column(str,   pa.Check.isin(["eletronicos", "alimentacao", "transporte"])),
        "valor":      pa.Column(float, pa.Check.greater_than_or_equal_to(0)),
        "cliente_id": pa.Column(str,   nullable=False),
    },
    strict=True,   # nenhuma coluna inesperada
)
print("Schema definido. Colunas exigidas:", list(schema.columns.keys()))

Schema definido. Colunas exigidas: ['id', 'data', 'categoria', 'valor', 'cliente_id']


### 2.1 — Validação que PASSA

Para mostrar o caminho feliz, validamos primeiro um **subconjunto limpo** (só as linhas que sabemos boas). O schema devolve o próprio DataFrame quando tudo passa.

In [5]:
limpo = vendas.loc[[0, 1, 2, 4, 8]].reset_index(drop=True)  # linhas sabidamente boas
validado = schema.validate(limpo)
print("Validacao PASSOU. Linhas validas:", len(validado))
validado

Validacao PASSOU. Linhas validas: 5


,id,data,categoria,valor,cliente_id
0,1001,2026-07-01,eletronicos,250.0,C001
1,1002,2026-07-01,alimentacao,45.5,C002
2,1003,2026-07-02,transporte,19.9,C003
3,1005,2026-07-03,alimentacao,12.0,C005
4,1009,2026-07-04,alimentacao,60.0,C009


### 2.2 — Falha PROPOSITAL, capturada com `lazy=True`

Agora validamos o `vendas` **completo**, que tem defeitos plantados. Com `lazy=True`, Pandera coleta **todas** as violações e as levanta num `SchemaErrors`. **Capturamos** a exceção (ela não propaga como erro de célula) e inspecionamos `failure_cases` — a tabela de todas as falhas, coluna por coluna.

In [6]:
falhas = None  # vai guardar o DataFrame failure_cases para a quarentena

try:
    schema.validate(vendas, lazy=True)
    print("Sem falhas (inesperado para este dataset).")
except pa.errors.SchemaErrors as e:
    falhas = e.failure_cases.copy()
    print(f"FALHA CAPTURADA: {len(falhas)} violacao(oes) detectada(s).\n")
    print("Resumo por (coluna, check):")
    resumo = (falhas.groupby(["column", "check"]).size()
                    .reset_index(name="n_violacoes"))
    print(resumo.to_string(index=False))

print("\n--- failure_cases (detalhe) ---")
falhas

FALHA CAPTURADA: 6 violacao(oes) detectada(s).

Resumo por (coluna, check):
    column                                              check  n_violacoes
 categoria isin(['eletronicos', 'alimentacao', 'transporte'])            1
cliente_id                                       not_nullable            1
      data                 str_matches('^\d{4}-\d{2}-\d{2}$')            1
        id                                   field_uniqueness            2
     valor                        greater_than_or_equal_to(0)            1

--- failure_cases (detalhe) ---


,schema_context,column,check,check_number,failure_case,index
0,Column,id,field_uniqueness,None,1003,2
1,Column,id,field_uniqueness,None,1003,9
2,Column,data,str_matches('^\d{4}-\d{2}-\d{2}$'),0,03/07/2026,6
3,Column,categoria,"isin(['eletronicos', 'alimentacao', 'transport...",0,viagem,7
4,Column,valor,greater_than_or_equal_to(0),0,-30.0,5
5,Column,cliente_id,not_nullable,None,None,3


Cada linha de `failure_cases` diz **qual coluna**, **qual check** foi violado e **qual valor** falhou — exatamente a informação que a quarentena vai usar como "motivo do erro".

---

# 3 — Great Expectations (com fallback)

> 📖 **No `.md`:** seção *3 — Great Expectations*.

GE é poderoso (suites, checkpoints, Data Docs) porém **pesado e instável entre versões**. Por isso ele é **opcional**: tentamos importá-lo; se não estiver disponível, o notebook **não quebra** — imprime o equivalente conceitual e segue. Nunca faça o pipeline depender de uma biblioteca instável para rodar.

In [7]:
try:
    import great_expectations as gx  # noqa: F401
    TEM_GE = True
    print("Great Expectations disponivel:", gx.__version__)
except Exception as e:
    TEM_GE = False
    print("Great Expectations indisponivel ->", type(e).__name__)

if TEM_GE:
    # Caminho real (API pode variar por versao; envolvido em try amplo)
    try:
        ctx = gx.get_context()
        validator = ctx.sources.pandas_default.read_dataframe(vendas)
        validator.expect_column_values_to_be_between("valor", min_value=0)
        validator.expect_column_values_to_be_in_set(
            "categoria", ["eletronicos", "alimentacao", "transporte"])
        validator.expect_column_values_to_not_be_null("cliente_id")
        resultado = validator.validate()
        print("Validacao GE concluida. success =", resultado["success"])
    except Exception as e:
        print("GE instalado, mas a API divergiu nesta versao:", type(e).__name__)
        TEM_GE = False

if not TEM_GE:
    print("=" * 64)
    print("FALLBACK — equivalente conceitual em Great Expectations")
    print("=" * 64)
    print("""
# Uma EXPECTATION SUITE e uma colecao de expectativas declarativas:
validator.expect_column_values_to_not_be_null("cliente_id")
validator.expect_column_values_to_be_between("valor", min_value=0)
validator.expect_column_values_to_be_in_set(
    "categoria", ["eletronicos", "alimentacao", "transporte"])
validator.expect_column_values_to_be_unique("id")

# Um CHECKPOINT executa a suite contra um lote, na hora definida
# (tipicamente disparado pelo orquestrador — tema do E6):
checkpoint = ctx.add_or_update_checkpoint(
    name="vendas_checkpoint", validator=validator)
checkpoint.run()   # passa / alerta / bloqueia o pipeline

# Os DATA DOCS sao paginas HTML geradas automaticamente a partir das
# execucoes: o que se esperava, o que passou, o que falhou — a
# 'documentacao que nao mente', porque nasce da propria validacao.
ctx.build_data_docs()
""")
    print("Conceitos: Expectation Suite (regras) -> Checkpoint (execucao) -> Data Docs (relatorio).")

Great Expectations indisponivel -> ModuleNotFoundError
FALLBACK — equivalente conceitual em Great Expectations

# Uma EXPECTATION SUITE e uma colecao de expectativas declarativas:
validator.expect_column_values_to_not_be_null("cliente_id")
validator.expect_column_values_to_be_between("valor", min_value=0)
validator.expect_column_values_to_be_in_set(
    "categoria", ["eletronicos", "alimentacao", "transporte"])
validator.expect_column_values_to_be_unique("id")

# Um CHECKPOINT executa a suite contra um lote, na hora definida
# (tipicamente disparado pelo orquestrador — tema do E6):
checkpoint = ctx.add_or_update_checkpoint(
    name="vendas_checkpoint", validator=validator)
checkpoint.run()   # passa / alerta / bloqueia o pipeline

# Os DATA DOCS sao paginas HTML geradas automaticamente a partir das
# execucoes: o que se esperava, o que passou, o que falhou — a
# 'documentacao que nao mente', porque nasce da propria validacao.
ctx.build_data_docs()

Conceitos: Expectation Suite (regras

---

# 4 — Data contract: contrato como código

> 📖 **No `.md`:** seção *4 — Data contracts*.

Um **data contract** é o "contrato de saída" do E1 virado código: schema + SLAs + semântica, acordado entre produtor e consumidor. Expressamos o contrato como um dicionário declarativo e mostramos que a parte de **schema** dele já é exatamente o que o Pandera valida.

In [8]:
contrato_vendas = {
    "dataset": "vendas_tratado",
    "versao": "1.0.0",
    "owner": "time-vendas",
    "schema": {
        "id":         {"tipo": "int",   "regras": ["> 0", "unico"], "pk": True},
        "data":       {"tipo": "str",   "regras": ["formato AAAA-MM-DD"]},
        "categoria":  {"tipo": "str",   "regras": ["in {eletronicos, alimentacao, transporte}"]},
        "valor":      {"tipo": "float", "regras": [">= 0"], "semantica": "valor bruto em BRL"},
        "cliente_id": {"tipo": "str",   "regras": ["nao-nulo"], "semantica": "FK para dim_cliente"},
    },
    "slas": {
        "frequencia": "diaria, ate 06:00",
        "completude_minima": {"cliente_id": 0.99, "valor": 1.0},
        "pontualidade_max_atraso_h": 2,
    },
    "semantica": "Uma linha = uma transacao de venda confirmada. 'valor' e bruto, "
                 "antes de impostos. 'categoria' segue a taxonomia oficial v3.",
}

print("CONTRATO DE DADOS — vendas_tratado v1.0.0")
print("-" * 50)
print("Schema (3 camadas: schema + SLAs + semantica):")
for col, regra in contrato_vendas["schema"].items():
    print(f"  {col:<12}: {regra['tipo']:<6} regras={regra['regras']}")
print(f"\nSLA frequencia : {contrato_vendas['slas']['frequencia']}")
print(f"SLA completude : {contrato_vendas['slas']['completude_minima']}")
print(f"\nO schema deste contrato e exatamente o DataFrameSchema da Secao 2 —")
print("o mesmo artefato serve produtor (compromisso), consumidor (expectativa) e pipeline (enforcement).")

CONTRATO DE DADOS — vendas_tratado v1.0.0
--------------------------------------------------
Schema (3 camadas: schema + SLAs + semantica):
  id          : int    regras=['> 0', 'unico']
  data        : str    regras=['formato AAAA-MM-DD']
  categoria   : str    regras=['in {eletronicos, alimentacao, transporte}']
  valor       : float  regras=['>= 0']
  cliente_id  : str    regras=['nao-nulo']

SLA frequencia : diaria, ate 06:00
SLA completude : {'cliente_id': 0.99, 'valor': 1.0}

O schema deste contrato e exatamente o DataFrameSchema da Secao 2 —
o mesmo artefato serve produtor (compromisso), consumidor (expectativa) e pipeline (enforcement).


---

# 5 — Quarentena: separar o ruim do bom

> 📖 **No `.md`:** seção *5 — Estratégias de tratamento*.

Em vez de **fail-fast** (abortar o lote inteiro por causa de poucas linhas ruins) ou **tolerante** (deixar a sujeira passar), aplicamos a **quarentena**: separamos válidas de inválidas. Os índices que aparecem em `failure_cases` (da Seção 2) são as linhas inválidas; o complemento é o conjunto válido. Enriquecemos a quarentena com o **motivo** do erro.

In [9]:
# Indices das linhas que falharam (a coluna 'index' de failure_cases aponta a linha original)
idx_invalidos = sorted(set(int(i) for i in falhas["index"].dropna().unique()))
print("Linhas invalidas (indice):", idx_invalidos)

# Motivo agregado por linha: junta todos os (coluna: check) que falharam naquela linha
motivos = (
    falhas.dropna(subset=["index"])
          .assign(motivo=lambda d: d["column"].astype(str) + " viola " + d["check"].astype(str))
          .groupby("index")["motivo"]
          .apply(lambda s: "; ".join(sorted(set(s))))
)

# --- Particionar ---
vendas_quarentena = vendas.loc[idx_invalidos].copy()
vendas_quarentena["motivo_erro"] = vendas_quarentena.index.map(motivos)

vendas_validas = vendas.drop(index=idx_invalidos).reset_index(drop=True)

print(f"\nValidas    : {len(vendas_validas)} linhas  -> seguem para o consumidor")
print(f"Quarentena : {len(vendas_quarentena)} linhas  -> isoladas para reprocessamento")
taxa_q = len(vendas_quarentena) / len(vendas) * 100
print(f"Taxa de quarentena: {taxa_q:.1f}%  (metrica de qualidade do lote)")

Linhas invalidas (indice): [2, 3, 5, 6, 7, 9]

Validas    : 4 linhas  -> seguem para o consumidor
Quarentena : 6 linhas  -> isoladas para reprocessamento
Taxa de quarentena: 60.0%  (metrica de qualidade do lote)


In [10]:
print("=== VENDAS EM QUARENTENA (com motivo) ===")
vendas_quarentena[["id", "data", "categoria", "valor", "cliente_id", "motivo_erro"]]

=== VENDAS EM QUARENTENA (com motivo) ===


,id,data,categoria,valor,cliente_id,motivo_erro
2,1003,2026-07-02,transporte,19.9,C003,id viola field_uniqueness
3,1004,2026-07-02,eletronicos,999.0,None,cliente_id viola not_nullable
5,1006,2026-07-03,transporte,-30.0,C006,valor viola greater_than_or_equal_to(0)
6,1007,03/07/2026,eletronicos,120.0,C007,data viola str_matches('^\d{4}-\d{2}-\d{2}$')
7,1008,2026-07-04,viagem,80.0,C008,"categoria viola isin(['eletronicos', 'alimenta..."
9,1003,2026-07-05,transporte,22.0,C010,id viola field_uniqueness


In [11]:
print("=== VENDAS VALIDAS (entregaveis ao consumidor) ===")
vendas_validas

=== VENDAS VALIDAS (entregaveis ao consumidor) ===


,id,data,categoria,valor,cliente_id
0,1001,2026-07-01,eletronicos,250.0,C001
1,1002,2026-07-01,alimentacao,45.5,C002
2,1005,2026-07-03,alimentacao,12.0,C005
3,1009,2026-07-04,alimentacao,60.0,C009


As linhas em quarentena **não são lixo**: guardam o registro original *e* o motivo, prontas para **reprocessamento** assim que a fonte ou a regra for corrigida. Sem o motivo, ninguém saberia o que consertar.

---

# 6 — Observabilidade: o cartão de saúde do lote

> 📖 **No `.md`:** seção *6 — Observabilidade e lineage*.

Validar responde "este lote está válido agora?". **Observar** responde "o pipeline está saudável ao longo do tempo?". Consolidamos as métricas-chave deste lote num **cartão de saúde** — uma linha de métricas pronta para ser **anexada a uma série temporal** a cada execução, onde a detecção de anomalias vira "desvio do padrão da própria série".

In [12]:
import datetime as dt

cartao = {
    "execucao_em":        dt.datetime(2026, 7, 10, 6, 0, 0).isoformat(),
    "linhas_entrada":     len(vendas),
    "linhas_validas":     len(vendas_validas),
    "linhas_quarentena":  len(vendas_quarentena),
    "taxa_quarentena_pct": round(len(vendas_quarentena) / len(vendas) * 100, 1),
    "ids_duplicados":     int(vendas["id"].duplicated().sum()),
    "completude_cliente_id_pct": round(vendas["cliente_id"].notna().mean() * 100, 1),
    "valor_medio":        round(float(vendas_validas["valor"].mean()), 2),
}

# Confronto com o SLA do contrato (Secao 4)
sla_cli = contrato_vendas["slas"]["completude_minima"]["cliente_id"] * 100
status_sla = "OK" if cartao["completude_cliente_id_pct"] >= sla_cli else "ALERTA"

print("CARTAO DE SAUDE DO LOTE — vendas_tratado")
print("=" * 48)
for k, v in cartao.items():
    print(f"  {k:<28}: {v}")
print("=" * 48)
print(f"SLA completude cliente_id >= {sla_cli:.0f}%  ->  {status_sla} "
      f"({cartao['completude_cliente_id_pct']}%)")
print("\nNuma execucao real, esta linha e gravada a cada rodada: a serie")
print("temporal permite detectar anomalias (queda de completude, salto no")
print("volume) e ALERTAR o engenheiro ANTES do consumidor.")

# Demonstracao: serie temporal com 3 execucoes (2 historicas + a de hoje)
historico = pd.DataFrame([
    {"execucao_em": "2026-07-08", "taxa_quarentena_pct": 5.0,  "completude_cliente_id_pct": 99.5, "valor_medio": 70.10},
    {"execucao_em": "2026-07-09", "taxa_quarentena_pct": 4.5,  "completude_cliente_id_pct": 99.8, "valor_medio": 71.30},
    {"execucao_em": "2026-07-10", "taxa_quarentena_pct": cartao["taxa_quarentena_pct"],
     "completude_cliente_id_pct": cartao["completude_cliente_id_pct"], "valor_medio": cartao["valor_medio"]},
])
print("\n=== SERIE TEMPORAL DE QUALIDADE (observabilidade) ===")
print(historico.to_string(index=False))
print("\nObserve o SALTO na taxa de quarentena hoje vs. historico -> anomalia a investigar.")

CARTAO DE SAUDE DO LOTE — vendas_tratado
  execucao_em                 : 2026-07-10T06:00:00
  linhas_entrada              : 10
  linhas_validas              : 4
  linhas_quarentena           : 6
  taxa_quarentena_pct         : 60.0
  ids_duplicados              : 1
  completude_cliente_id_pct   : 90.0
  valor_medio                 : 91.88
SLA completude cliente_id >= 99%  ->  ALERTA (90.0%)

Numa execucao real, esta linha e gravada a cada rodada: a serie
temporal permite detectar anomalias (queda de completude, salto no
volume) e ALERTAR o engenheiro ANTES do consumidor.

=== SERIE TEMPORAL DE QUALIDADE (observabilidade) ===
execucao_em  taxa_quarentena_pct  completude_cliente_id_pct  valor_medio
 2026-07-08                  5.0                       99.5        70.10
 2026-07-09                  4.5                       99.8        71.30
 2026-07-10                 60.0                       90.0        91.88

Observe o SALTO na taxa de quarentena hoje vs. historico -> anomalia a in

---

## 🧪 Exercícios do Encontro 5

Use os objetos já criados acima (`vendas`, `schema`, `falhas`, `vendas_validas`, `vendas_quarentena`, `contrato_vendas`).

1. **Nova regra no schema.** Adicione ao `schema` um `Check` customizado que rejeite **datas no futuro** (`data > hoje`). Dica: use `pa.Check(lambda s: pd.to_datetime(s, errors="coerce") <= pd.Timestamp.today())` na coluna `data`. Revalide com `lazy=True` e veja se alguma linha nova cai na quarentena. Por que tratar a coluna `data` exige cuidado extra (lembre que uma das linhas tem data fora do formato)?

2. **Completude por coluna como gate.** Escreva uma função `gate_completude(df, minimos: dict) -> bool` que receba um dicionário `{coluna: completude_minima}` (como o do SLA no contrato) e retorne `False` se **qualquer** coluna ficar abaixo do mínimo, imprimindo quais violaram. Teste com `{"cliente_id": 0.99, "valor": 1.0}` sobre o `vendas`.

3. **Reprocessamento.** Simule a **correção** das linhas em quarentena: conserte o `cliente_id` ausente (linha 1004), a data fora de padrão (1007) e a categoria inválida (1008) — descarte de fato apenas o valor negativo (1006) e o id duplicado (1003). Reinjete as corrigidas, revalide e mostre quantas linhas passam agora a integrar `vendas_validas`.

4. **Métrica nova de observabilidade.** Acrescente ao `cartao` de saúde uma métrica de **distribuição**: o desvio-padrão de `valor` nas linhas válidas. Em seguida, escreva em 2–3 linhas como você usaria essa métrica, junto com o histórico, para disparar um alerta de anomalia **sem** que nenhuma linha individual seja "inválida".

## Encerramento

Hoje demos ao pipeline de vendas a sua **camada de qualidade**:

- **medimos** qualidade (completude por coluna, unicidade, validade) transformando "dado bom" em número;
- **validamos** com **Pandera** (`DataFrameSchema` + `Check`), com sucesso e com uma **falha proposital capturada** via `lazy=True`;
- esboçamos **Great Expectations** (suites, checkpoints, Data Docs) de forma resiliente, sem depender dele para rodar;
- escrevemos um **data contract** (schema + SLAs + semântica) — o contrato de saída do E1 virado código;
- aplicamos **quarentena**: separamos válidas de inválidas, anexando o **motivo** do erro, em vez de fail-fast ou tolerância cega;
- consolidamos um **cartão de saúde** e uma **série temporal** de observabilidade, para flagrar anomalias **antes** do consumidor.

O elo que costura tudo é o **shift-left**: validar cedo é barato; descobrir tarde custa credibilidade.

**No próximo encontro (E6 — Orquestração e automação de pipelines)**, fazemos tudo isto rodar **sozinho, na hora certa, avisando quando falha** — onde os *checkpoints* de validação e o cartão de saúde encontram o agendador.

---

*Notebook prático — UNIFOR Pós-Graduação · Prof. Cassio Pinheiro · Encontro 5 · 10/07/2026.*